# Running Models — Audio · Part 7 — Companion Notebook

This goes with **"Steering the Band: Length, Melody, and Sampling"**. Run cells top to bottom (Shift+Enter). For GPU: Runtime → Change runtime type → GPU.

## Setup

Install the libraries (skip if already installed in this session).

In [ ]:
!pip install -q transformers torchaudio

## Reload the band

Same small MusicGen checkpoint from Part 6.

In [ ]:
from transformers import AutoProcessor, MusicgenForConditionalGeneration
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to(device)
sr = model.config.audio_encoder.sampling_rate

inputs = processor(text=["a warm acoustic guitar loop"], padding=True, return_tensors="pt").to(device)

## Lever 1 — length, in tokens

Clip length is a token budget at ~50 tokens/sec: `seconds × 50`.

In [ ]:
audio = model.generate(**inputs, max_new_tokens=512)   # ~10 s at ~50 tokens/sec

from IPython.display import Audio
Audio(audio[0, 0].cpu().numpy(), rate=sr)

## Lever 2 — sampling, temperature, guidance

The same decoding knobs from phase-1 text generation, plus `guidance_scale`.

In [ ]:
audio = model.generate(
    **inputs,
    do_sample=True,        # sample tokens instead of greedy argmax -> more variety
    temperature=1.0,       # higher = wilder
    guidance_scale=3.0,    # classifier-free guidance: how strongly to follow the prompt
    max_new_tokens=256,
)
Audio(audio[0, 0].cpu().numpy(), rate=sr)

## Reproducing a take

Seed the RNG before `generate` to get the same clip back every time.

In [ ]:
torch.manual_seed(0)
audio = model.generate(**inputs, do_sample=True, guidance_scale=3.0, max_new_tokens=256)
Audio(audio[0, 0].cpu().numpy(), rate=sr)

## Lever 4 — batching several prompts

Pass a list of prompts; `padding=True` is required so uneven prompts pack into one tensor.

In [ ]:
inputs = processor(
    text=["a tense cinematic build-up", "a calm acoustic guitar loop"],
    padding=True,
    return_tensors="pt",
).to(device)

audios = model.generate(**inputs, do_sample=True, guidance_scale=3.0, max_new_tokens=256)
# audios[0] and audios[1] are the two clips
Audio(audios[0, 0].cpu().numpy(), rate=sr)

In [ ]:
Audio(audios[1, 0].cpu().numpy(), rate=sr)

## Lever 5 — melody conditioning (heavier, separate model)

**Note:** this is a different, heavier checkpoint (`musicgen-melody`) with its own class — `musicgen-small` does not take audio. The melody API is **version-sensitive**: if the import or the `processor(...)` call errors on your installed `transformers`, check the model card and docs for your version.

In [ ]:
from transformers import AutoProcessor, MusicgenMelodyForConditionalGeneration
import torchaudio, torch

processor = AutoProcessor.from_pretrained("facebook/musicgen-melody")
model = MusicgenMelodyForConditionalGeneration.from_pretrained("facebook/musicgen-melody").to(device)

melody, msr = torchaudio.load(
    torchaudio.utils.download_asset("tutorial-assets/Lab41-SRI-VOiCES-src-sp0307-ch127535-sg0042.wav")
)

inputs = processor(
    audio=melody[0].numpy(),
    sampling_rate=msr,
    text=["jazzy piano version"],
    padding=True,
    return_tensors="pt",
).to(device)

audio = model.generate(**inputs, max_new_tokens=256)
Audio(audio[0, 0].cpu().numpy(), rate=model.config.audio_encoder.sampling_rate)

## Where next

Part 8 wraps all of this into a small prompt-to-soundtrack app.